In [10]:
import numpy as np
import pandas as pd
import time
import os
import json

from data.datasets import selecionar_dataset_e_classe, carregar_dataset
from utils.results_handler import update_method_results
from utils.progress_bar import ProgressBar

from tqdm import tqdm  # Alterado de tqdm.notebook para usar a versão de texto, evitando o erro de IProgress
# Imports do Scikit-learn para o novo experimento
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.neural_network import MLPClassifier  # MLP
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance # <-- Para a nova heurística

RANDOM_STATE = 42


In [11]:
# Constantes e funções de configuração reutilizadas do peab.py

# Configurações específicas de MNIST (opcional, mas bom ter para consistência)
MNIST_CONFIG = {
    'feature_mode': 'raw',
    'digit_pair': (3, 8),
    'top_k_features': None,
    'test_size': 0.3,
    'rejection_cost': 0.24,
    'subsample_size': 0.01
}

# Dicionário principal de configuração dos datasets
DATASET_CONFIG = {
    "mnist":                MNIST_CONFIG,
    "breast_cancer":        {'test_size': 0.3, 'rejection_cost': 0.24},
    "pima_indians_diabetes":{'test_size': 0.3, 'rejection_cost': 0.24},
    "vertebral_column":     {'test_size': 0.3, 'rejection_cost': 0.24},
    "sonar":                {'test_size': 0.3, 'rejection_cost': 0.24},
    "spambase":             {'test_size': 0.3, 'rejection_cost': 0.24},
    "banknote":             {'test_size': 0.3, 'rejection_cost': 0.24},
    "heart_disease":        {'test_size': 0.3, 'rejection_cost': 0.24},
    "wine":                 {'subsample_size': 0.20, 'test_size': 0.3, 'rejection_cost': 0.24},
    "creditcard":           {'subsample_size': 0.03, 'test_size': 0.3, 'rejection_cost': 0.040},
    "covertype":            {'subsample_size': 0.005, 'test_size': 0.3, 'rejection_cost': 0.24},
    "gas_sensor":           {'subsample_size': 0.05, 'test_size': 0.3, 'rejection_cost': 0.045},
    "newsgroups":           {'subsample_size': 0.5, 'test_size': 0.3, 'rejection_cost': 0.24},
    "rcv1":                 {'subsample_size': 0.5, 'test_size': 0.3, 'rejection_cost': 0.24},
}

def configurar_experimento(dataset_name: str):
    """Carrega o dataset e as configurações específicas para ele."""
    if dataset_name == 'mnist':
        from data import datasets as ds_module
        cfg = DATASET_CONFIG.get(dataset_name, {})
        ds_module.set_mnist_options(cfg.get('feature_mode', 'raw'), cfg.get('digit_pair', None))
    
    X, y, nomes_classes = carregar_dataset(dataset_name)
    cfg = DATASET_CONFIG.get(dataset_name, {'test_size': 0.3, 'rejection_cost': 0.24})

    return X, y, nomes_classes, cfg['rejection_cost'], cfg['test_size']

In [12]:
# --- SELEÇÃO DO DATASET ---
# 1. Liste os datasets disponíveis
available_datasets = list(DATASET_CONFIG.keys())
print("Datasets disponíveis:")
for i, name in enumerate(available_datasets):
    print(f"  {i}: {name}")

# 2. Escolha o dataset pelo número (índice)
dataset_index = 0 # <-- Mude aqui! (ex: 1 para breast_cancer, 2 para pima_indians_diabetes)
DATASET_NAME = available_datasets[dataset_index]
# --- FIM DA SELEÇÃO ---

print(f"\n--> Dataset selecionado: '{DATASET_NAME}'\n")

X, y, nomes_classes, rejection_cost, test_size = configurar_experimento(DATASET_NAME)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=RANDOM_STATE, stratify=y
)

print(f"Dataset '{DATASET_NAME}' carregado.")
print(f"Treino: {X_train.shape[0]} instâncias | Teste: {X_test.shape[0]} instâncias")

Datasets disponíveis:
  0: mnist
  1: breast_cancer
  2: pima_indians_diabetes
  3: vertebral_column
  4: sonar
  5: spambase
  6: banknote
  7: heart_disease
  8: wine
  9: creditcard
  10: covertype
  11: gas_sensor
  12: newsgroups
  13: rcv1

--> Dataset selecionado: 'mnist'

Dataset 'mnist' carregado.
Treino: 9776 instâncias | Teste: 4190 instâncias


In [13]:
def treinar_modelo_mlp(X_train, y_train, mlp_params):
    """
    Cria e treina um pipeline com MinMaxScaler e MLPClassifier.
    """
    pipeline = Pipeline([
        ('scaler', MinMaxScaler()),
        ('model', MLPClassifier(random_state=RANDOM_STATE, **mlp_params))
    ])
    
    print("Treinando o modelo MLP...")
    start_time = time.time()
    pipeline.fit(X_train, y_train)
    end_time = time.time()
    print(f"Treinamento concluído em {end_time - start_time:.2f} segundos.")
    
    return pipeline

# Parâmetros MLP.
MLP_PARAMS = {
    'hidden_layer_sizes': (100, 50),  # Duas camadas ocultas com 100 e 50 neurônios
    'activation': 'relu',
    'solver': 'adam',
    'max_iter': 500,                 # numero maximo de epocas que o modelo pode treinar
    'alpha': 0.0001,                 # Termo de regularização L2
    'learning_rate_init': 0.001,     # taxa de aprendizado incial
    'early_stopping': True,          # True para parar quando o backpropagation não conseguir mais ajustar
    'n_iter_no_change': 10,          # Nº de iterações sem melhora antes de parar
    'validation_fraction': 0.1       # Fração do treino usada para validação do early stopping
}

# Treinando o modelo para ter um objeto para os próximos passos
print("Treinando o modelo MLP... (para ter o objeto 'modelo_mlp' disponível)")
modelo_mlp = treinar_modelo_mlp(X_train, y_train, MLP_PARAMS)

# A lógica de encontrar os thresholds (t+ e t-) continua a mesma,
# basta usar o `modelo_mlp.decision_function()` no seu conjunto de validação.


Treinando o modelo MLP... (para ter o objeto 'modelo_mlp' disponível)
Treinando o modelo MLP...
Treinamento concluído em 3.82 segundos.


In [14]:
def check_validity_model_agnostic(
    fixed_indices: set,
    instance_df: pd.DataFrame,
    modelo: Pipeline,
    X_train: pd.DataFrame, # Usado para obter os limites min/max
    t_plus: float,
    t_minus: float,
    mode: str
) -> bool:
    """
    Verifica a validade de uma explicação para um modelo "caixa-preta" (como MLP).

    Esta função perturba as features não fixadas para o pior caso e
    executa o predict do modelo para verificar se a decisão se mantém.
    É mais lenta que a versão linear, mas funciona para qualquer modelo.
    """
    scaler = modelo.named_steps['scaler']
    
    # Pega os limites (min/max) de cada feature do treino no espaço original
    min_vals_original = X_train.min().values
    max_vals_original = X_train.max().values
    
    # Cria duas cópias da instância para os piores cenários
    perturbed_instance_min_case = instance_df.iloc[0].copy()
    perturbed_instance_max_case = instance_df.iloc[0].copy()

    # Para cada feature que NÃO está na explicação, a jogamos para seu pior valor
    for i in range(len(X_train.columns)):
        if i not in fixed_indices:
            # Para um modelo não-linear, não sabemos a direção do "pior caso" a priori.
            # Uma heurística robusta é testar os dois extremos: o mínimo e o máximo global.
            # Isso simula um adversário que pode mover as features livremente.
            perturbed_instance_min_case.iloc[i] = min_vals_original[i]
            perturbed_instance_max_case.iloc[i] = max_vals_original[i]

    # Converte para DataFrame para o pipeline
    df_min_case = pd.DataFrame([perturbed_instance_min_case])
    df_max_case = pd.DataFrame([perturbed_instance_max_case])

    # Roda a predição do MLP para os piores casos
    # Usamos predict_proba e calculamos o log-odds ratio como score
    # score = log(P(classe=1) / P(classe=0))
    epsilon = 1e-9
    probas_min = np.clip(modelo.predict_proba(df_min_case)[0], epsilon, 1 - epsilon)
    probas_max = np.clip(modelo.predict_proba(df_max_case)[0], epsilon, 1 - epsilon)

    score_min_case = np.log(probas_min[1] / probas_min[0])
    score_max_case = np.log(probas_max[1] / probas_max[0])

    # Valida de acordo com o modo
    EPSILON = 1e-5
    if mode == 'positive':
        # Para ser positivo, mesmo no pior cenário possível (que pode ser o min ou o max),
        # o score tem que ficar acima de t+. Testamos o menor score obtido.
        worst_score = min(score_min_case, score_max_case)
        return worst_score >= t_plus - EPSILON
        
    elif mode == 'negative':
        # Para ser negativo, mesmo no pior cenário, o score tem que ficar abaixo de t-.
        # Testamos o maior score obtido.
        best_adversary_score = max(score_min_case, score_max_case)
        return best_adversary_score <= t_minus + EPSILON
        
    elif mode == 'rejected':
        # [CORREÇÃO DO BUG] Para ser rejeitado, o score tem que permanecer na zona de rejeição
        # mesmo nos piores cenários adversários. A lógica anterior estava incorreta.
        # O menor score possível (pior caso do adversário para empurrar para baixo) não pode cair abaixo de t-.
        # O maior score possível (pior caso do adversário para empurrar para cima) não pode subir acima de t+.
        worst_score = min(score_min_case, score_max_case)
        best_adversary_score = max(score_min_case, score_max_case)
        
        return (worst_score >= t_minus - EPSILON) and (best_adversary_score <= t_plus + EPSILON)
        
    return False


In [15]:
def calcular_sorting_metric_mlp(modelo: Pipeline, X_val: pd.DataFrame, y_val: pd.Series) -> np.ndarray:
    """
    Calcula uma métrica de importância de features para a MLP usando
    Permutation Importance. Se falhar por memória, usa heurística de pesos.
    """
    print("Calculando a importância das features por permutação (pode levar um momento)...")
    start_time = time.time()
    
    try:
        # [CORREÇÃO] n_jobs=1 é crucial no Windows para evitar MemoryError com datasets grandes (MNIST)
        # Reduzimos n_repeats para 5 para economizar tempo e memória sem perder muita precisão
        result = permutation_importance(
            modelo, X_val, y_val, 
            n_repeats=5, 
            random_state=RANDOM_STATE, 
            n_jobs=1,  # Serializado para economizar RAM
            scoring='accuracy' # Garante uso da métrica correta
        )
        importancias = result.importances_mean
        
    except MemoryError:
        print("⚠️ Erro de memória no Permutation Importance. Usando heurística de pesos da 1ª camada.")
        # Fallback: Heurística rápida (Produto dos pesos da entrada pela média dos pesos da saída)
        # Acessa o modelo MLP dentro do pipeline
        mlp = modelo.named_steps['model'] if 'model' in modelo.named_steps else modelo
        
        # Pega pesos da primeira camada (Input -> Hidden)
        # Shape: (n_features, n_hidden)
        w1 = mlp.coefs_[0] 
        
        # Média absoluta dos pesos conectando cada feature (proxy de importância)
        importancias = np.mean(np.abs(w1), axis=1)
    
    end_time = time.time()
    print(f"Cálculo concluído em {end_time - start_time:.2f} segundos.")
    
    return importancias

# Exemplo de uso (usando o próprio X_test como validação aqui, mas o ideal é um split separado)
sorting_metric_mlp = calcular_sorting_metric_mlp(modelo_mlp, X_test, y_test)

# Visualizando as features mais importantes
feature_importances = pd.Series(sorting_metric_mlp, index=X_train.columns).sort_values(ascending=False)
print("\nTop 10 features mais importantes segundo a Permutation Importance:")
print(feature_importances.head(10))


Calculando a importância das features por permutação (pode levar um momento)...


KeyboardInterrupt: 

In [ ]:
def encontrar_thresholds_otimos(X_train, y_train, rejection_cost, mlp_params, val_size=0.2):
    """
    Encontra os thresholds t+ e t- ótimos em um conjunto de validação.
    """
    print("Encontrando thresholds ótimos (t+ e t-)...")
    X_train_sub, X_val, y_train_sub, y_val = train_test_split(
        X_train, y_train, test_size=val_size, random_state=RANDOM_STATE, stratify=y_train
    )
    
    # Treina um modelo temporário no sub-conjunto de treino
    modelo_temp = treinar_modelo_mlp(X_train_sub, y_train_sub, mlp_params)
    
    # Calcula scores no conjunto de validação
    # Usamos predict_proba e calculamos o log-odds ratio como score
    probas = modelo_temp.predict_proba(X_val)
    epsilon = 1e-9
    probas = np.clip(probas, epsilon, 1 - epsilon)
    decision_scores = np.log(probas[:, 1] / probas[:, 0])
    
    scores_neg = decision_scores[decision_scores < 0]
    scores_pos = decision_scores[decision_scores > 0]
    
    t_minus_grid = np.linspace(scores_neg.min(), -0.0001, 50) if len(scores_neg) > 0 else np.array([-0.1])
    t_plus_grid = np.linspace(0.0001, scores_pos.max(), 50) if len(scores_pos) > 0 else np.array([0.1])
    
    best_risk = float('inf')
    best_t_plus, best_t_minus = 0.1, -0.1
    
    for tm in t_minus_grid:
        for tp in t_plus_grid:
            if not (tm < 0 < tp): continue
            
            preds = np.full(y_val.shape, -1) # -1 para rejeitado
            accepted_mask = (decision_scores >= tp) | (decision_scores <= tm)
            
            preds[decision_scores >= tp] = 1
            preds[decision_scores <= tm] = 0
            
            error = np.mean(preds[accepted_mask] != y_val.values[accepted_mask]) if np.any(accepted_mask) else 0.0
            rejection_rate = 1.0 - np.mean(accepted_mask)
            
            risk = error + rejection_cost * rejection_rate
            
            if risk < best_risk:
                best_risk, best_t_plus, best_t_minus = risk, tp, tm
                
    print(f"Thresholds encontrados: t+={best_t_plus:.4f}, t-={best_t_minus:.4f} (Risco: {best_risk:.4f})")
    return float(best_t_plus), float(best_t_minus)

# Encontrar os thresholds para o nosso modelo
t_plus, t_minus = encontrar_thresholds_otimos(X_train, y_train, rejection_cost, MLP_PARAMS)

Encontrando thresholds ótimos (t+ e t-)...
Treinando o modelo MLP...
Treinamento concluído em 3.94 segundos.
Thresholds encontrados: t+=2.9606, t-=-2.1147 (Risco: 0.0097)


In [ ]:
def fase_1_reforco_agnostic(modelo: Pipeline, instance_df: pd.DataFrame, X_train: pd.DataFrame, t_plus: float, t_minus: float, mode: str, sorting_metric: np.ndarray) -> set:
    expl_indices = set()
    indices_ordenados = np.argsort(-sorting_metric)
    
    for idx in indices_ordenados:
        # Verifica se a explicação já é válida
        if check_validity_model_agnostic(expl_indices, instance_df, modelo, X_train, t_plus, t_minus, mode):
            break
        # Adiciona a feature mais importante que ainda não está na explicação
        if idx not in expl_indices:
            expl_indices.add(idx)
            
    return expl_indices

def fase_2_minimizacao_agnostic(modelo: Pipeline, instance_df: pd.DataFrame, expl_indices_inicial: set, X_train: pd.DataFrame, t_plus: float, t_minus: float, mode: str, sorting_metric: np.ndarray) -> set:
    expl_indices = expl_indices_inicial.copy()
    # Ordena as features da explicação para tentar remover as de menor importância primeiro
    features_presentes = list(expl_indices)
    features_presentes.sort(key=lambda i: sorting_metric[i], reverse=False)
    
    for idx in features_presentes:
        if len(expl_indices) <= 1: break
        
        # Tenta remover a feature
        expl_indices.remove(idx)
        
        # Se a explicação se tornar inválida, desfaz a remoção
        if not check_validity_model_agnostic(expl_indices, instance_df, modelo, X_train, t_plus, t_minus, mode):
            expl_indices.add(idx)
            
    return expl_indices

def gerar_explicacao_instancia_mlp(instancia_df: pd.DataFrame, modelo: Pipeline, X_train: pd.DataFrame, t_plus: float, t_minus: float, sorting_metric: np.ndarray) -> list:
    # Usamos predict_proba e calculamos o log-odds ratio como score
    probas = modelo.predict_proba(instancia_df)[0]
    epsilon = 1e-9
    probas = np.clip(probas, epsilon, 1 - epsilon)
    score_raw = np.log(probas[1] / probas[0])
    
    feature_names = X_train.columns

    if score_raw >= t_plus:
        mode = 'positive'
    elif score_raw <= t_minus:
        mode = 'negative'
    else:
        mode = 'rejected'

    # Fase 1: Adicionar features até a explicação se tornar robusta
    indices_robustos = fase_1_reforco_agnostic(modelo, instancia_df, X_train, t_plus, t_minus, mode, sorting_metric)
    
    # Fase 2: Remover features redundantes da explicação robusta
    indices_minimos = fase_2_minimizacao_agnostic(modelo, instancia_df, indices_robustos, X_train, t_plus, t_minus, mode, sorting_metric)

    return [feature_names[i] for i in sorted(list(indices_minimos))]

In [ ]:
# --- CÉLULA DE EXECUÇÃO E COLETA DE DADOS ---

print(f"Iniciando a geração de explicações para {X_test.shape[0]} instâncias de teste...")
start_time_total = time.time()

# Estrutura para armazenar resultados por instância
per_instance_results = []

# Pré-cálculo das predições e scores para todo o conjunto de teste
# Usamos predict_proba e calculamos o log-odds ratio como score
probas = modelo_mlp.predict_proba(X_test)
epsilon = 1e-9
probas = np.clip(probas, epsilon, 1 - epsilon)
scores = np.log(probas[:, 1] / probas[:, 0])

preds_sem_rejeicao = modelo_mlp.predict(X_test)

# Determinar predições com rejeição (0: neg, 1: pos, 2: rej)
preds_com_rejeicao = np.full(len(X_test), 2)
preds_com_rejeicao[scores >= t_plus] = 1
preds_com_rejeicao[scores <= t_minus] = 0

# Loop principal com barra de progresso
for i in tqdm(range(len(X_test)), desc="Gerando Explicações MLP"):
    inst_df = X_test.iloc[[i]]
    start_inst_time = time.time()
    
    # Gerar explicação
    explicacao = gerar_explicacao_instancia_mlp(inst_df, modelo_mlp, X_train, t_plus, t_minus, sorting_metric_mlp)
    
    end_inst_time = time.time()
    
    # Armazenar resultados da instância
    per_instance_results.append({
        'id': str(X_test.index[i]),
        'y_true': int(y_test.iloc[i]),
        'y_pred_sem_rejeicao': int(preds_sem_rejeicao[i]),
        'y_pred_com_rejeicao': int(preds_com_rejeicao[i]),
        'decision_score': float(scores[i]),
        'explicacao': explicacao,
        'tamanho_explicacao': len(explicacao),
        'tempo_execucao': end_inst_time - start_inst_time
    })

end_time_total = time.time()
total_execution_time = end_time_total - start_time_total
print(f"\nGeração de explicações concluída em {total_execution_time:.2f} segundos.")

# --- AGREGAÇÃO E SALVAMENTO EM JSON ---

# Calcular estatísticas agregadas
mask_rej = (preds_com_rejeicao == 2)
acc_sem_rejeicao = np.mean(preds_sem_rejeicao == y_test) * 100
acc_com_rejeicao = np.mean(preds_com_rejeicao[~mask_rej] == y_test.values[~mask_rej]) * 100 if np.any(~mask_rej) else 100.0

tamanhos_pos = [r['tamanho_explicacao'] for r in per_instance_results if r['y_pred_com_rejeicao'] == 1]
tamanhos_neg = [r['tamanho_explicacao'] for r in per_instance_results if r['y_pred_com_rejeicao'] == 0]
tamanhos_rej = [r['tamanho_explicacao'] for r in per_instance_results if r['y_pred_com_rejeicao'] == 2]

def calc_stats(data_list):
    if not data_list: return {'count': 0, 'mean': 0, 'std': 0, 'min': 0, 'max': 0}
    return {
        'count': len(data_list),
        'mean': float(np.mean(data_list)),
        'std': float(np.std(data_list)),
        'min': int(np.min(data_list)),
        'max': int(np.max(data_list))
    }

# Montar o dicionário final
final_results = {
    'config': {
        'dataset_name': DATASET_NAME,
        'num_instances_total': X.shape[0],
        'num_features': X.shape[1],
        'test_size': test_size,
        'rejection_cost': rejection_cost,
        'random_state': RANDOM_STATE
    },
    'model_params': {
        'model_type': 'MLPClassifier',
        'hidden_layer_sizes': MLP_PARAMS['hidden_layer_sizes'],
        'activation': MLP_PARAMS['activation'],
        'solver': MLP_PARAMS['solver'],
        'max_iter': MLP_PARAMS['max_iter'],
        'n_iter_': modelo_mlp.named_steps['model'].n_iter_, # Épocas reais
        'n_layers_': modelo_mlp.named_steps['model'].n_layers_,
    },
    'thresholds': {
        't_plus': t_plus,
        't_minus': t_minus,
        'rejection_zone_width': t_plus - t_minus
    },
    'performance': {
        'accuracy_without_rejection': acc_sem_rejeicao,
        'accuracy_with_rejection': acc_com_rejeicao,
        'rejection_rate': np.mean(mask_rej) * 100,
        'num_test_instances': len(X_test),
        'num_positive': len(tamanhos_pos),
        'num_negative': len(tamanhos_neg),
        'num_rejected': len(tamanhos_rej),
    },
    'explanation_stats': {
        'positive': calc_stats(tamanhos_pos),
        'negative': calc_stats(tamanhos_neg),
        'rejected': calc_stats(tamanhos_rej)
    },
    'computation_time': {
        'total_seconds': total_execution_time,
        'mean_per_instance_seconds': total_execution_time / len(X_test) if len(X_test) > 0 else 0
    },
    'per_instance': per_instance_results
}

# Salvar em JSON
output_dir_json = 'json/MLP'
os.makedirs(output_dir_json, exist_ok=True)
json_filepath = os.path.join(output_dir_json, f'{DATASET_NAME}.json')

with open(json_filepath, 'w', encoding='utf-8') as f:
    json.dump(final_results, f, indent=4)

print(f"Resultados salvos em: {json_filepath}")

Iniciando a geração de explicações para 4190 instâncias de teste...


Gerando Explicações MLP:   0%|          | 17/4190 [06:06<25:01:17, 21.59s/it]


KeyboardInterrupt: 

In [ ]:
# --- CÉLULA DE GERAÇÃO DE RELATÓRIO A PARTIR DO JSON ---

def gerar_relatorio_texto_mlp(json_filepath: str):
    """
    Gera um relatório de texto formatado a partir de um arquivo JSON de resultados do MLP.
    """
    # Carregar dados do JSON
    with open(json_filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    # Criar diretório de saída
    output_dir_report = 'results/report_MLP'
    os.makedirs(output_dir_report, exist_ok=True)
    report_filepath = os.path.join(output_dir_report, f"report_{data['config']['dataset_name']}.txt")

    # Extrair seções do dicionário para facilitar o acesso
    cfg = data['config']
    model = data['model_params']
    thresh = data['thresholds']
    perf = data['performance']
    exp_stats = data['explanation_stats']
    comp_time = data['computation_time']

    with open(report_filepath, 'w', encoding='utf-8') as f:
        f.write("="*80 + "\n")
        f.write(f"          RELATÓRIO DE ANÁLISE - MÉTODO MLP COM REJEIÇÃO\n")
        f.write("="*80 + "\n\n")

        # 1. Resumo do Experimento
        f.write("1. RESUMO DO EXPERIMENTO\n")
        f.write("-"*80 + "\n")
        f.write(f"  - Nome do Dataset: {cfg['dataset_name']}\n")
        f.write(f"  - Total de Instâncias: {cfg['num_instances_total']}\n")
        f.write(f"  - Total de Features: {cfg['num_features']}\n")
        f.write(f"  - Divisão Treino/Teste: {1-cfg['test_size']:.0%}/{cfg['test_size']:.0%}\n")
        f.write(f"  - Instâncias de Treino: {cfg['num_instances_total'] - perf['num_test_instances']}\n")
        f.write(f"  - Instâncias de Teste: {perf['num_test_instances']}\n\n")

        # 2. Configuração do Modelo MLP
        f.write("2. CONFIGURAÇÃO DO MODELO (MLP)\n")
        f.write("-"*80 + "\n")
        f.write(f"  - Neurônios nas Camadas Ocultas: {model['hidden_layer_sizes']}\n")
        f.write(f"  - Função de Ativação: {model['activation']}\n")
        f.write(f"  - Solver: {model['solver']}\n")
        f.write(f"  - Épocas de Treinamento (Backpropagation): {model['n_iter_']}\n\n")

        # 3. Zona de Rejeição
        f.write("3. ZONA DE REJEIÇÃO\n")
        f.write("-"*80 + "\n")
        f.write(f"  - Custo de Rejeição (usado para otimização): {cfg['rejection_cost']}\n")
        f.write(f"  - Limiar Superior (t+): {thresh['t_plus']:.4f}\n")
        f.write(f"  - Limiar Inferior (t-): {thresh['t_minus']:.4f}\n")
        f.write(f"  - Tamanho da Zona de Rejeição: {thresh['rejection_zone_width']:.4f}\n\n")

        # 4. Desempenho
        f.write("4. DESEMPENHO DO CLASSIFICADOR\n")
        f.write("-"*80 + "\n")
        f.write(f"  - Acurácia ANTES da Rejeição: {perf['accuracy_without_rejection']:.2f}%\n")
        f.write(f"  - Acurácia DEPOIS da Rejeição (nas aceitas): {perf['accuracy_with_rejection']:.2f}%\n")
        f.write(f"  - Taxa de Rejeição: {perf['rejection_rate']:.2f}%\n")
        f.write(f"  - Instâncias Positivas: {perf['num_positive']}\n")
        f.write(f"  - Instâncias Negativas: {perf['num_negative']}\n")
        f.write(f"  - Instâncias Rejeitadas: {perf['num_rejected']}\n\n")

        # 5. Estatísticas das Explicações
        f.write("5. ESTATÍSTICAS DAS EXPLICAÇÕES\n")
        f.write("-"*80 + "\n")
        f.write(f"  - Para Instâncias POSITIVAS ({exp_stats['positive']['count']}):\n")
        f.write(f"    - Tamanho Médio da Explicação: {exp_stats['positive']['mean']:.2f} features\n")
        f.write(f"  - Para Instâncias NEGATIVAS ({exp_stats['negative']['count']}):\n")
        f.write(f"    - Tamanho Médio da Explicação: {exp_stats['negative']['mean']:.2f} features\n")
        f.write(f"  - Para Instâncias REJEITADAS ({exp_stats['rejected']['count']}):\n")
        f.write(f"    - Tamanho Médio da Explicação: {exp_stats['rejected']['mean']:.2f} features\n\n")

        # 6. Tempo de Execução
        f.write("6. TEMPO DE EXECUÇÃO (Geração das Explicações)\n")
        f.write("-"*80 + "\n")
        f.write(f"  - Tempo Total: {comp_time['total_seconds']:.2f} segundos\n")
        f.write(f"  - Tempo Médio por Instância: {comp_time['mean_per_instance_seconds']:.4f} segundos\n\n")
        
    print(f"Relatório de texto gerado em: {report_filepath}")

# Chamar a função para gerar o relatório usando o arquivo JSON salvo anteriormente
gerar_relatorio_texto_mlp(json_filepath)

Relatório de texto gerado em: results/report_MLP\report_breast_cancer.txt
